In [1]:
import pandas as pd

# --- 1. Load core files ---
# Movies metadata
movies = pd.read_csv(
    "data/ml-100k/u.item",
    sep="|", encoding="latin-1",
    names=["movie_id","title","release_date","video_release_date","IMDb_URL",
           "unknown","Action","Adventure","Animation","Children","Comedy","Crime",
           "Documentary","Drama","Fantasy","Film-Noir","Horror","Musical","Mystery",
           "Romance","Sci-Fi","Thriller","War","Western"]
)

# Ratings
ratings = pd.read_csv(
    "data/ml-100k/u.data",
    sep="\t", names=["user_id","item_id","rating","timestamp"]
)

# Users (optional enrichment)
users = pd.read_csv(
    "data/ml-100k/u.user",
    sep="|", names=["user_id","age","gender","occupation","zip"]
)

# --- 2. Collapse one-hot genre flags into a genre_list ---
genre_cols = ["Action","Adventure","Animation","Children","Comedy","Crime",
              "Documentary","Drama","Fantasy","Film-Noir","Horror","Musical",
              "Mystery","Romance","Sci-Fi","Thriller","War","Western"]

def collapse_genres(row):
    return [g for g in genre_cols if row[g] == 1]

movies["genre_list"] = movies.apply(collapse_genres, axis=1)

# --- 3. Enrich with ratings ---
# Compute average rating and rating count per movie
ratings_summary = ratings.groupby("item_id").agg(
    avg_rating=("rating","mean"),
    rating_count=("rating","count")
).reset_index().rename(columns={"item_id":"movie_id"})

# Merge into movies
movies_enriched = movies.merge(ratings_summary, on="movie_id", how="left")

# --- 4. Select lean schema ---
movies_final = movies_enriched[[
    "movie_id",
    "title",
    "release_date",
    "genre_list",
    "avg_rating",
    "rating_count"
]]

   movie_id              title release_date                     genre_list  \
0         1   Toy Story (1995)  01-Jan-1995  [Animation, Children, Comedy]   
1         2   GoldenEye (1995)  01-Jan-1995  [Action, Adventure, Thriller]   
2         3  Four Rooms (1995)  01-Jan-1995                     [Thriller]   
3         4  Get Shorty (1995)  01-Jan-1995        [Action, Comedy, Drama]   
4         5     Copycat (1995)  01-Jan-1995       [Crime, Drama, Thriller]   

   avg_rating  rating_count  
0    3.878319           452  
1    3.206107           131  
2    3.033333            90  
3    3.550239           209  
4    3.302326            86  


In [2]:
# Total rows
total_rows = len(movies_final)

# Count missing and non-missing per column
missing_count = movies_final.isnull().sum()
non_missing_count = total_rows - missing_count
missing_percent = (missing_count / total_rows) * 100

# Combine into one DataFrame
missing_summary = pd.DataFrame({
    "non_missing": non_missing_count,
    "missing": missing_count,
    "missing_percent": missing_percent.round(2)
})

print(missing_summary)

              non_missing  missing  missing_percent
movie_id             1682        0             0.00
title                1682        0             0.00
release_date         1681        1             0.06
genre_list           1682        0             0.00
avg_rating           1682        0             0.00
rating_count         1682        0             0.00
